# Notebook 06: Master Data Management (MDM) y Esquema Estrella Relacional
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Gold $\rightarrow$ Relational OLAP Star Schema  
**Estándar**: Ralph Kimball / DAMA-DMBOK 2 (Master & Reference Data) / SWEBOK v4  

---
### Objetivos del Modelo Estrella y MDM
1. **Reconciliación Maestra (MDM)**: Normalización difusa de entidades geográficas (municipios DIVIPOLA) y cadenas productivas agrícolas mediante métricas combinadas de Levenshtein y Jaro-Winkler con umbral $\ge 0.85$.
2. **Aprovisionamiento DDL**: Creación idempotente del modelo dimensional estrella en SQLite / DuckDB.
3. **Poblado Dimensional**: Carga de dimensiones maestras (`dim_tiempo`, `dim_municipio`, `dim_producto`, `dim_insumo`).
4. **Carga de Tablas de Hechos**: Transacción segura de hechos agrícolas (`fact_produccion_eva`), climáticos (`fact_clima_ideam`) y de precios (`fact_precios_sipsa`).
5. **Explotación OLAP**: Consultas analíticas desnormalizadas sobre `view_indicadores_agroclimaticos`.

In [1]:
# Celda 1: Setup e Importaciones de MDM y Base de Datos
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.database.mdm_manager import MDMManager, levenshtein_similarity, jaro_winkler_similarity
from src.database.db_manager import DatabaseManager
from src.database.sql_loader import SQLLoader

logger = get_logger("notebooks.06_database_mdm")
db_mgr = DatabaseManager(use_duckdb=False)
loader = SQLLoader(db_manager=db_mgr)
mdm = MDMManager(match_threshold=0.85)
print(f"DatabaseManager inicializado: {db_mgr.db_file}")

### 1. Reconciliación Difusa de Entidades Maestras (MDM)

In [2]:
# Celda 2: Demostración de algoritmos de coincidencia difusa
df_divipola_sample = pd.DataFrame([
    {"cod_municipio": "05001", "nombre_municipio": "MEDELLIN", "cod_departamento": "05", "nombre_departamento": "ANTIOQUIA"},
    {"cod_municipio": "11001", "nombre_municipio": "BOGOTA, D.C.", "cod_departamento": "11", "nombre_departamento": "BOGOTA D.C."},
    {"cod_municipio": "76001", "nombre_municipio": "CALI", "cod_departamento": "76", "nombre_departamento": "VALLE DEL CAUCA"},
    {"cod_municipio": "68001", "nombre_municipio": "BUCARAMANGA", "cod_departamento": "68", "nombre_departamento": "SANTANDER"},
])
mdm.register_canonical_municipios(df_divipola_sample)

pruebas_nombres = ["Medellin", "Bogota", "Santiago de Cali", "Bucaramanga D.E."]
resultados_mdm = []
for nombre in pruebas_nombres:
    match = mdm.match_municipio(nombre)
    resultados_mdm.append({
        "Nombre Entrada": nombre,
        "Golden Record": match['nombre_municipio'] if match else 'NO MATCH',
        "Código DIVIPOLA": match['cod_municipio'] if match else 'N/A',
        "Score Similitud": match['similarity_score'] if match else 0.0,
        "Tipo Match": match['match_type'] if match else 'RECHAZADO'
    })

display(pd.DataFrame(resultados_mdm))

### 2. Aprovisionamiento DDL del Esquema Estrella Relacional

In [3]:
# Celda 3: Inicialización de tablas y vistas dimensionales
db_mgr.initialize_schema()
df_tablas = db_mgr.execute_query("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name;")
print("Tablas y Vistas creadas en el modelo estrella:")
display(df_tablas)

### 3. Carga Masiva de Dimensiones Maestras (`dim_tiempo`, `dim_municipio`)

In [4]:
# Celda 4: Inserción de dimensiones base
n_tiempo = loader.load_dim_tiempo(start_year=2018, end_year=2025)
n_mun = loader.load_dim_municipio(df_divipola_sample)
print(f"Dimensiones pobladas: dim_tiempo={n_tiempo} períodos | dim_municipio={n_mun} municipios")

df_sample_tiempo = db_mgr.execute_query("SELECT * FROM dim_tiempo LIMIT 5;")
display(df_sample_tiempo)

### 4. Carga de Hechos Agrícolas (`fact_produccion_eva`)

In [5]:
# Celda 5: Carga de hechos desde Silver hacia la tabla de hechos
eva_silver_path = settings.CLEANED_DIR / "evaluaciones_agropecuarias_eva_silver.parquet"
if eva_silver_path.exists():
    df_eva = pd.read_parquet(eva_silver_path)
else:
    df_eva = pd.DataFrame([
        {"anio": 2022, "cod_municipio": "05001", "cultivo": "Café", "variedad": "Caturra", "area_sembrada_ha": 120.0, "area_cosechada_ha": 115.0, "produccion_ton": 230.0, "rendimiento_ton_ha": 2.0},
        {"anio": 2022, "cod_municipio": "11001", "cultivo": "Papa", "variedad": "Pastusa", "area_sembrada_ha": 80.0, "area_cosechada_ha": 80.0, "produccion_ton": 1200.0, "rendimiento_ton_ha": 15.0},
    ])

filas_cargadas = loader.load_fact_eva(df_eva)
print(f"Registros insertados en fact_produccion_eva: {filas_cargadas}")

### 5. Consultas Analíticas OLAP sobre `view_indicadores_agroclimaticos`

In [6]:
# Celda 6: Ejecución de consultas analíticas multidimensionales
query_olap = """
SELECT 
    anio,
    nombre_municipio,
    nombre_producto,
    variedad,
    SUM(produccion_ton) as total_produccion_ton,
    AVG(rendimiento_ton_ha) as promedio_rendimiento_t_ha
FROM view_indicadores_agroclimaticos
GROUP BY anio, nombre_municipio, nombre_producto, variedad
ORDER BY total_produccion_ton DESC;
"""
df_olap = db_mgr.execute_query(query_olap)
print("Resultados agregados desde la vista dimensional OLAP:")
display(df_olap)